# Теория. Модель машинного обучения как веб-сервис
## «Модель обучена. Как ей воспользоваться водителю, диспетчеру и мобильному приложению?»

**Цель:** модель из итоговой работы Темы 1 отвечает на запросы по сети, проверяет входные данные и записывает прогнозы в журнал. После занятия вы сохраняете и загружаете конвейер scikit-learn, понимаете, почему модель загружают один раз при старте сервера, проверяете признаки вручную и моделью Pydantic, отвечаете классом и вероятностями и храните прогнозы в столбце `jsonb` базы PostgreSQL.

**Как работать с ноутбуком.** Ячейки выполняются по порядку (Shift+Enter). Ноутбук лежит рядом с папкой `ПР_ML_сервис` и берет из нее файл модели. Разделы 1–8 работают без серверов. Разделу 5 (вторая половина) нужен FastAPI, разделам 9 и 10 – запущенный сервер PostgreSQL и файл `.env` в папке `ПР_ML_сервис/server` (как в Т4.Б1.4), разделу 11 – запущенный сервер приложения.

**План:**
1. Где живет модель после обучения
2. Сохранение и загрузка конвейера: joblib
3. Почему нельзя загружать чужие файлы pickle
4. Версии библиотек при обучении и в сервисе
5. Загрузка модели один раз при старте
6. Проверка входных данных вручную
7. Модель Pydantic для признаков
8. Прогноз: класс, вероятности, версия, время
9. Журнал прогнозов: тип `jsonb` в PostgreSQL
10. Обратная связь и доля верных прогнозов
11. Сервис целиком: запросы клиента
12. Итоги, словарь и вопросы

## 1. Где живет модель после обучения

В итоговой работе Темы 1 модель жила в ноутбуке: обучили, проверили на тестовой выборке, написали отчет. Пользоваться ей мог только тот, кто открыл ноутбук. Путь модели к пользователям:

| Где модель | Кто может ей пользоваться | Что нужно |
| --- | --- | --- |
| переменная в ноутбуке | автор, пока ноутбук открыт | ничего |
| файл `model.joblib` | любая программа на Python с теми же библиотеками | сохранить и загрузить конвейер |
| веб-сервис | любая программа на любом языке, по сети | программный интерфейс поверх модели |

Кто будет вызывать сервис «Покрытие»:
- **страница на React** (Блок 2) – диспетчер вводит показания и видит вероятности;
- **мобильное приложение на Kotlin** (Тема 5) – водитель получает прогноз в машине;
- **другая программа** – например, бортовой компьютер присылает показания датчиков каждую секунду.

Никому из них не нужно знать, что внутри логистическая регрессия, и ставить scikit-learn. Им нужен **контракт**: какие признаки отправить, в каком формате придет ответ, какие бывают ошибки. Контракт описывают модели Pydantic, а FastAPI показывает его на странице `/docs`.

```
клиент ──POST /api/predict {14 признаков}──> FastAPI ──> проверка (Pydantic) ──> модель ──> журнал (PostgreSQL)
клиент <──200 {класс, вероятности, версия}── FastAPI
клиент <──422 {"field": "slip", ...}──────── FastAPI     (если данные неверные, модель не вызывается)
```

## 2. Сохранение и загрузка конвейера: joblib

Обученный объект Python можно записать в файл и потом восстановить. Для этого в Python есть модуль `pickle` («консервирование»), а библиотека `joblib` работает так же, но быстрее для больших массивов numpy. Документация scikit-learn рекомендует именно ее:

```python
joblib.dump(объект, "model.joblib")     # сохранить
объект = joblib.load("model.joblib")    # загрузить
```

Сохранять нужно **весь конвейер** `Pipeline`, а не только логистическую регрессию. Модель видела стандартизованные признаки и закодированные категории: без медиан, средних и списка категорий из обучающей выборки ее ответы бессмысленны.

Преподаватель сохранил не голый конвейер, а словарь: конвейер и **паспорт модели** – версию, список признаков, классы, версию scikit-learn и метрики (скрипт `model/train_model.py`). Посмотрим, что в файле.

In [1]:
import json
import pathlib
import sys
import time

import joblib
import numpy as np
import pandas as pd
import sklearn

SERVER = pathlib.Path("ПР_ML_сервис") / "server"     # папка сервера рядом с ноутбуком
sys.path.insert(0, str(SERVER))                      # чтобы импортировать модули из app
MODEL_FILE = SERVER / "model" / "model.joblib"

bundle = joblib.load(MODEL_FILE)
print("ключи:", list(bundle))
for key in ["version", "description", "trained_at", "train_rows", "sklearn_version", "metrics"]:
    print(f"{key:16}", bundle[key])
print("признаков:", len(bundle["features"]), bundle["features"])
print("классы:", bundle["classes"])

ключи: ['pipeline', 'version', 'description', 'features', 'classes', 'trained_at', 'train_rows', 'metrics', 'sklearn_version', 'python_version']
version          logreg-1.0
description      Логистическая регрессия из итоговой работы Темы 1, вариант 1
trained_at       2026-11-06
train_rows       1000
sklearn_version  1.9.1
metrics          {'C': 1, 'cv_f1_macro': 0.877, 'test_f1_macro': 0.893, 'test_recall_snow': 0.933}
признаков: 14 ['speed', 'vib_front', 'vib_rear', 'vib_hf', 'slip', 'air_temp', 'road_temp', 'humidity', 'lidar_intensity', 'camera_brightness', 'cabin_noise', 'wipers', 'time_of_day', 'tires']
классы: ['грунт', 'мокрый асфальт', 'снег и лед', 'сухой асфальт']


In [2]:
pipeline = bundle["pipeline"]
print(pipeline)
print()
print("размер файла:", MODEL_FILE.stat().st_size, "байт")

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  ['speed', 'vib_front',
                                                   'vib_rear', 'vib_hf', 'slip',
                                                   'air_temp', 'road_temp',
                                                   'humidity',
                                                   'lidar_intensity',
                                                   'camera_brightness',
                                                   'cabin_noise']),
                                                 ('cat',

Внутри конвейера то же, что вы собирали в итоговой работе: `ColumnTransformer` заполняет пропуски и стандартизует 11 числовых признаков, кодирует 3 категориальных, затем логистическая регрессия. Файл весит около 6 КБ: логистической регрессии нужно хранить только веса, медианы и средние.

Проверим, что модель после загрузки отвечает. Конвейер обучен на таблице pandas, поэтому на вход ему нужна таблица с теми же столбцами, даже если строка одна.

In [3]:
reference = json.loads((SERVER / "tests" / "data" / "reference.json").read_text(encoding="utf-8"))
example = reference["examples"][0]
print("признаки:", example["features"])
print("настоящее покрытие:", example["label"])

frame = pd.DataFrame([example["features"]], columns=bundle["features"])   # таблица из одной строки
print("ответ:", pipeline.predict(frame)[0])
print("вероятности:", pipeline.predict_proba(frame)[0].round(4))

признаки: {'speed': 44.8, 'vib_front': 0.358, 'vib_rear': 0.367, 'vib_hf': 0.339, 'slip': 2.12, 'air_temp': 20.4, 'road_temp': 29.9, 'humidity': 63, 'lidar_intensity': 0.515, 'camera_brightness': 131, 'cabin_noise': 62.3, 'wipers': 'выкл', 'time_of_day': 'день', 'tires': 'летние'}
настоящее покрытие: сухой асфальт
ответ: сухой асфальт
вероятности: [0.0112 0.0017 0.     0.9872]


Как сохраняют и проверяют файл (так сделано в `train_model.py`): после `dump` сразу загрузить и сравнить ответы с исходным объектом на нескольких примерах.

In [4]:
import tempfile

with tempfile.TemporaryDirectory() as folder:
    path = pathlib.Path(folder) / "copy.joblib"
    joblib.dump(bundle, path)
    copy = joblib.load(path)

examples = pd.DataFrame([e["features"] for e in reference["examples"]], columns=bundle["features"])
same = np.allclose(pipeline.predict_proba(examples), copy["pipeline"].predict_proba(examples))
print("ответы копии совпадают с оригиналом:", same)

ответы копии совпадают с оригиналом: True


## 3. Почему нельзя загружать чужие файлы pickle

`pickle` и `joblib` хранят не данные, а **инструкции, как собрать объект заново**. Среди инструкций может быть вызов любой функции Python. Поэтому документация Python предупреждает в первых строках описания модуля `pickle`: «Модуль pickle небезопасен. Загружайте только данные, которым доверяете». Злоумышленник может собрать файл, при загрузке которого выполнится его код.

Безобидный опыт. Объект класса `Trap` при сохранении говорит: «чтобы восстановить меня, вызови функцию `print` с таким текстом». Вместо `print` могла быть функция, которая удаляет файлы или отправляет по сети ваш `.env`.

In [5]:
import pickle

class Trap:
    def __reduce__(self):
        # Инструкция для восстановления объекта: (функция, аргументы). Злоумышленник подставит что угодно.
        return (print, ("Этот текст напечатал код, записанный в файле. Модель еще даже не загрузилась.",))

data = pickle.dumps(Trap())      # «файл модели», который прислал незнакомец
print("размер:", len(data), "байт")
pickle.loads(data)               # загрузка выполняет код из файла

размер: 180 байт
Этот текст напечатал код, записанный в файле. Модель еще даже не загрузилась.


Правила:
1. Загружайте только модели, которые сохранили вы или ваш преподаватель, из своего репозитория.
2. Не скачивайте файлы `.pkl` и `.joblib` с форумов и из чужих репозиториев.
3. Для обмена моделями между людьми есть безопасные форматы без исполняемых инструкций: например, `skops.io` для scikit-learn и ONNX для разных библиотек. В курсе мы их не используем: модель у нас своя.

## 4. Версии библиотек при обучении и в сервисе

Файл `joblib` не хранит код scikit-learn, только состояние объектов: веса, медианы, имена параметров. При загрузке объекты собираются **классами из установленной версии библиотеки**. Если между версиями поменялось внутреннее устройство класса, возможны три исхода:
1. файл не загрузится (ошибка) – неприятно, но честно;
2. загрузится с предупреждением `InconsistentVersionWarning` – scikit-learn сам сравнивает версии;
3. загрузится и будет отвечать иначе – хуже всего: этого никто не заметит.

Поэтому:
- версии библиотек модели зафиксированы в `requirements.txt`: `scikit-learn==1.9.1`, `joblib`, `numpy`, `pandas` – те же, что при обучении;
- версия scikit-learn записана в паспорте модели, а сервис при старте сравнивает ее с установленной и отказывается работать с чужой версией (задание З1). Лучше не запуститься с понятной ошибкой, чем молча давать неверные прогнозы беспилотному автомобилю.

In [6]:
print("модель сохранена в scikit-learn", bundle["sklearn_version"])
print("установлена версия           ", sklearn.__version__)
print("совпадают:", bundle["sklearn_version"] == sklearn.__version__)
print("Python при обучении:", bundle["python_version"], " сейчас:", sys.version.split()[0])

модель сохранена в scikit-learn 1.9.1
установлена версия            1.9.1
совпадают: True
Python при обучении: 3.13.18  сейчас: 3.13.18


Если версии в школе другие (например, в зеркале пакетов нет нужной), модель переобучают той версией, которая есть: `python model/train_model.py`. Для этого и хранится скрипт обучения рядом с моделью: файл модели без кода, который его сделал, нельзя воспроизвести.

## 5. Загрузка модели один раз при старте

Сравним, сколько стоит загрузка модели и сколько – один прогноз. Часы для замеров – `time.perf_counter()`: они точнее `time.time()` и не зависят от перевода системного времени.

In [7]:
def ms(seconds):
    return f"{seconds * 1000:.2f} мс"

start = time.perf_counter()
loaded = joblib.load(MODEL_FILE)
load_time = time.perf_counter() - start

row = pd.DataFrame([example["features"]], columns=loaded["features"])
loaded["pipeline"].predict_proba(row)                    # первый прогноз «прогревает» модель
times = []
for _ in range(50):
    start = time.perf_counter()
    loaded["pipeline"].predict_proba(row)
    times.append(time.perf_counter() - start)

print("загрузка файла (модули уже загружены):", ms(load_time))
print("один прогноз, медиана из 50:           ", ms(sorted(times)[25]))

загрузка файла (модули уже загружены): 0.84 мс
один прогноз, медиана из 50:            3.26 мс


Повторная загрузка нашей модели быстрая: файл маленький, а модули scikit-learn ноутбук уже загрузил выше. В свежем процессе Python (так стартует сервер) первая загрузка вместе с модулями занимает около 60 мс, то есть в 15–20 раз дольше прогноза. А нейросети из Темы 3 весят сотни мегабайт, и их загрузка занимает секунды. Если загружать модель в обработчике, каждый запрос платит эту цену заново и держит в памяти свою копию модели.

Решение: загрузить модель **один раз при старте сервера** и хранить в памяти приложения. В FastAPI для этого есть **функция жизненного цикла** `lifespan`: код до `yield` выполняется при старте, код после `yield` – при остановке. Хранилище для общих объектов приложения – `app.state`.

```python
from contextlib import asynccontextmanager

@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.model = load_model(settings.model_path)   # при старте, до первого запроса
    yield                                                # сервер принимает запросы
    # здесь можно освободить ресурсы при остановке

app = FastAPI(lifespan=lifespan)

@app.post("/api/predict")
def predict(features: Features, request: Request):
    bundle = request.app.state.model                     # готовая модель, без загрузки
```

`lifespan` – единственное место в нашем коде с `async def`: FastAPI требует, чтобы функция жизненного цикла была асинхронным менеджером контекста. Загрузка внутри нее обычная, синхронная.

Опыт ниже (нужен FastAPI) показывает, что загрузка происходит один раз, сколько бы запросов ни пришло. `TestClient` в блоке `with` запускает и останавливает приложение, как настоящий сервер.

In [ ]:
from contextlib import asynccontextmanager
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient

loads = []

@asynccontextmanager
async def demo_lifespan(app):
    loads.append(time.perf_counter())
    print("старт: загружаю модель")
    app.state.model = joblib.load(MODEL_FILE)
    yield
    print("остановка")

demo = FastAPI(lifespan=demo_lifespan)

@demo.get("/version")
def version(request: Request):
    return {"version": request.app.state.model["version"]}

with TestClient(demo) as client:
    for _ in range(3):
        print(client.get("/version").json())
print("сколько раз загружена модель:", len(loads))

## 6. Проверка входных данных вручную

Модель ответит на **любые** числа, даже бессмысленные. Возьмем пример с сухим асфальтом и подставим в температуру дороги `-999` – код ошибки датчика, который мы нашли в итоговой работе Темы 1.

In [8]:
CODES = {"сухой асфальт": "dry", "мокрый асфальт": "wet", "грунт": "dirt", "снег и лед": "snow"}

def answer(features):
    frame = pd.DataFrame([features], columns=bundle["features"])
    proba = pipeline.predict_proba(frame)[0]
    return {CODES[name]: round(float(p), 3) for name, p in zip(pipeline.classes_, proba)}

broken = {**example["features"], "road_temp": -999}
print("исходные признаки:", answer(example["features"]))
print("road_temp = -999: ", answer(broken))

исходные признаки: {'dirt': 0.011, 'wet': 0.002, 'snow': 0.0, 'dry': 0.987}
road_temp = -999:  {'dirt': 0.0, 'wet': 0.0, 'snow': 1.0, 'dry': 0.0}


Модель уверенно «видит» лед на сухом асфальте: при стандартизации `-999` превращается в огромное отрицательное число, и логистическая регрессия доводит вероятность снега почти до 1. Беспилотный автомобиль резко затормозит на сухой дороге. Поэтому сервис обязан **проверять входные данные до модели** и отвечать понятной ошибкой с кодом 422.

Три вида ошибок входных данных:

| Ошибка | Пример | Что ответить |
| --- | --- | --- |
| признака нет | клиент забыл `speed` | 422, «обязательное поле» |
| неверный тип | `"slip": "2.12"`, `"humidity": true`, `"tires": "шипы"` | 422, «должно быть числом» или список допустимых значений |
| значение вне диапазона | `"road_temp": -999`, `"air_temp": 150` | 422, допустимый диапазон |

Описание, единицы и диапазоны всех 14 признаков лежат в готовом словаре `NUMERIC_SPECS` (модуль `app/features.py`): это правила очистки из итоговой работы и физический смысл датчиков.

In [9]:
from app.features import CATEGORY_SPECS, NUMERIC_SPECS

for name, (description, unit, low, high) in NUMERIC_SPECS.items():
    print(f"{name:18} {low:>4} … {high:<4} {unit:10} {description}")
for name, (description, allowed) in CATEGORY_SPECS.items():
    print(f"{name:18} {', '.join(allowed):26} {description}")

speed                 0 … 200  км/ч       Скорость автомобиля
vib_front             0 … 20   м/с²       Вертикальная вибрация передней подвески, среднеквадратичное ускорение за секунду
vib_rear              0 … 20   м/с²       Вертикальная вибрация задней подвески
vib_hf                0 … 20   м/с²       Высокочастотная вибрация кузова
slip                  0 … 50   %          Проскальзывание ведущих колес относительно скорости по спутниковой навигации
air_temp            -60 … 60   °C         Температура воздуха
road_temp           -60 … 80   °C         Температура поверхности дороги по инфракрасному датчику
humidity              0 … 100  %          Относительная влажность воздуха
lidar_intensity       0 … 1    от 0 до 1  Средняя интенсивность отражения лидара от дороги перед автомобилем
camera_brightness     0 … 255  от 0 до 255 Средняя яркость нижней части кадра передней камеры
cabin_noise          20 … 120  дБ         Уровень шума в салоне
wipers             вкл, выкл             

Проверка одного числового признака вручную. Две ловушки:
- **`True` – это число?** В Python `bool` – подкласс `int`: `isinstance(True, int)` дает `True`. Значение `true` из JSON прошло бы как 1. Логическое значение надо проверять отдельно и раньше.
- **«Не число» NaN.** `float("nan")` не меньше и не больше никакого числа, поэтому проверка `value < low or value > high` его пропустит. Проверка `not low <= value <= high` – нет.

In [10]:
def check_number(data, name):
    # Проверка одного числового признака. Возвращает текст ошибки или None.
    _, _, low, high = NUMERIC_SPECS[name]
    if name not in data:
        return "обязательное поле"
    value = data[name]
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        return "должно быть числом"
    if not low <= value <= high:
        return f"должно быть от {low} до {high}"
    return None

print("True – int?", isinstance(True, int))
for value in [29.9, 30, "29.9", True, -999, float("nan"), None]:
    print(f"road_temp = {value!r:8} ->", check_number({"road_temp": value}, "road_temp"))
print("нет признака ->", check_number({}, "road_temp"))

nan = float("nan")
print("NaN < -60 or NaN > 80:", nan < -60 or nan > 80, " not (-60 <= NaN <= 80):", not (-60 <= nan <= 80))

True – int? True
road_temp = 29.9     -> None
road_temp = 30       -> None
road_temp = '29.9'   -> должно быть числом
road_temp = True     -> должно быть числом
road_temp = -999     -> должно быть от -60 до 80
road_temp = nan      -> должно быть от -60 до 80
road_temp = None     -> должно быть числом
нет признака -> обязательное поле
NaN < -60 or NaN > 80: False  not (-60 <= NaN <= 80): True


Для 14 признаков такая проверка занимает пару десятков строк с ветвлениями (задание З2). Ошибки нужно собирать все сразу, а не останавливаться на первой: клиенту удобнее исправить все за один раз. И главный недостаток ручной проверки: о диапазонах клиент узнает только из ошибок, в документации `/docs` их нет.

## 7. Модель Pydantic для признаков

Ту же проверку можно **объявить**, а не написать: модель Pydantic с полем на каждый признак (задание З3).

| Параметр `Field` | Смысл |
| --- | --- |
| `ge=0`, `le=200` | больше или равно, меньше или равно: границы диапазона |
| `strict=True` | строгий режим: строка `"12.5"` и логическое `true` не превращаются в число |
| `description="..."` | описание, которое попадет в документацию `/docs` и `/openapi.json` |

Параметр модели `extra="forbid"` (в `ConfigDict`) запрещает лишние поля: клиент, который пришлет `rec_speed`, получит ошибку и узнает, что модель этот признак не использует. По умолчанию Pydantic лишние поля молча пропускает.

Небольшая модель из трех признаков для опыта:

In [11]:
from enum import StrEnum
from pydantic import BaseModel, ConfigDict, Field, ValidationError

class Tires(StrEnum):
    summer = "летние"
    winter = "зимние"

class Demo(BaseModel):
    model_config = ConfigDict(extra="forbid")

    road_temp: float = Field(strict=True, ge=-60, le=80, description="Температура поверхности дороги, °C")
    humidity: float = Field(strict=True, ge=0, le=100, description="Относительная влажность воздуха, %")
    tires: Tires = Field(description="Тип шин: летние или зимние")

print(Demo(road_temp=29.9, humidity=63, tires="летние"))

bad = {"road_temp": -999, "humidity": "63", "tires": "шипы", "rec_speed": 90}
try:
    Demo.model_validate(bad)
except ValidationError as error:
    for problem in error.errors():
        print(problem["loc"], problem["type"], "|", problem["msg"])

road_temp=29.9 humidity=63.0 tires=<Tires.summer: 'летние'>
('road_temp',) greater_than_equal | Input should be greater than or equal to -60
('humidity',) float_type | Input should be a valid number
('tires',) enum | Input should be 'летние' or 'зимние'
('rec_speed',) extra_forbidden | Extra inputs are not permitted


Одна ошибка на каждое нарушение, все сразу, с местом (`loc`) и видом (`type`). Вид ошибки обработчик исключений из Т4.Б1.3 (модуль `app/errors.py`) переводит в русский текст единого формата `{"error": ..., "field": ..., "errors": [...]}`.

Целое число для поля `float` в строгом режиме допустимо: `63` – то же, что `63.0`. А почему строгий режим включен только у числовых полей? FastAPI проверяет уже разобранный JSON, в котором значение перечисления – обычная строка `"летние"`. В строгом режиме Pydantic потребовал бы объект `Tires`, а не строку.

Без строгого режима Pydantic пытается превратить значение в нужный тип:

In [12]:
class Lax(BaseModel):
    humidity: float = Field(ge=0, le=100)

print("без strict:", Lax(humidity="63"), Lax(humidity=True))
try:
    Demo(road_temp=10, humidity="63", tires="летние")
except ValidationError as error:
    print("со strict: ", error.errors()[0]["type"])

без strict: humidity=63.0 humidity=1.0
со strict:  float_type


Для формы на странице такое превращение удобно, а для программного интерфейса скрывает ошибки клиента: он шлет числа строками, и все работает, пока однажды не придет `"12,5"` с запятой.

Описание модели для документации Pydantic строит сам. Именно это FastAPI покажет на странице `/docs` и отдаст разработчику мобильного приложения:

In [13]:
schema = Demo.model_json_schema()
print("обязательные:", schema["required"])
print("лишние поля разрешены:", schema["additionalProperties"])
print(json.dumps(schema["properties"]["road_temp"], ensure_ascii=False))

обязательные: ['road_temp', 'humidity', 'tires']
лишние поля разрешены: False
{"description": "Температура поверхности дороги, °C", "maximum": 80, "minimum": -60, "title": "Road Temp", "type": "number"}


| | Ручная проверка (З2) | Модель Pydantic (З3) |
| --- | --- | --- |
| объем кода | ветвления на каждый вид ошибки | одна строка на признак |
| все ошибки сразу | надо собирать самому | да |
| ответ 422 в FastAPI | надо вызывать самому | автоматически, до обработчика |
| описание в `/docs` | нет | да: типы, диапазоны, описания, пример |
| лишние поля | надо проверять самому | `extra="forbid"` |

## 8. Прогноз: класс, вероятности, версия, время

Ответ сервиса (модель ответа `PredictionOut`):

```json
{"id": 17, "surface": "dry", "surface_name": "сухой асфальт",
 "probabilities": {"dry": 0.9872, "wet": 0.0017, "dirt": 0.0112, "snow": 0.0},
 "model_version": "logreg-1.0", "elapsed_ms": 1.84, "created_at": "2026-11-06T10:15:02+07:00"}
```

Зачем каждое поле:
- **класс** – ответ на вопрос;
- **вероятности всех классов** – уверенность модели: при 0,48 и 0,23 у двух классов ответ ненадежен, и клиент может предупредить водителя; для безопасности важна и вероятность опасного класса отдельно;
- **версия модели** – модель будут переобучать, и надо знать, какая версия дала этот ответ;
- **время обработки** – наблюдение за скоростью сервиса (подробно в Т4.Б3.2);
- **номер и время записи** – ссылка на запись журнала, по которой водитель пришлет обратную связь (раздел 10).

**Ловушка с порядком классов.** `predict_proba` возвращает вероятности в порядке `pipeline.classes_`, а scikit-learn сортирует классы по алфавиту. Этот порядок не совпадает ни с порядком из итоговой работы, ни с порядком кодов `dry, wet, dirt, snow`.

In [14]:
print("pipeline.classes_:", list(pipeline.classes_))
proba = pipeline.predict_proba(pd.DataFrame([example["features"]], columns=bundle["features"]))[0]
print("вероятности:      ", proba.round(4))

ORDER = ["dry", "wet", "dirt", "snow"]
wrong = {code: round(float(p), 4) for code, p in zip(ORDER, proba)}          # ошибка: чужой порядок
right = {CODES[name]: round(float(p), 4) for name, p in zip(pipeline.classes_, proba)}
print("неверно:", wrong, "-> ответ", max(wrong, key=wrong.get))
print("верно:  ", {code: right[code] for code in ORDER}, "-> ответ", max(right, key=right.get))
print("настоящее покрытие:", example["label"])

pipeline.classes_: ['грунт', 'мокрый асфальт', 'снег и лед', 'сухой асфальт']
вероятности:       [0.0112 0.0017 0.     0.9872]
неверно: {'dry': 0.0112, 'wet': 0.0017, 'dirt': 0.0, 'snow': 0.9872} -> ответ snow
верно:   {'dry': 0.9872, 'wet': 0.0017, 'dirt': 0.0112, 'snow': 0.0} -> ответ dry
настоящее покрытие: сухой асфальт


С неверным сопоставлением сервис ответил бы «снег и лед» на сухом асфальте, хотя модель уверена в сухом асфальте. Тесты З4 сравнивают ответы с эталоном и такую ошибку поймают.

Весь прогноз для сервиса – одна функция (задание З4): таблица из одной строки, вероятности, перевод в коды покрытия, округление. Время меряем вокруг нее.

In [15]:
def predict_one(bundle, features):
    frame = pd.DataFrame([features], columns=bundle["features"])
    proba = bundle["pipeline"].predict_proba(frame)[0]
    probabilities = {CODES[name]: float(p) for name, p in zip(bundle["pipeline"].classes_, proba)}
    best = max(probabilities, key=probabilities.get)
    return best, {code: round(probabilities[code], 4) for code in ORDER}

start = time.perf_counter()
surface, probabilities = predict_one(bundle, example["features"])
elapsed_ms = round((time.perf_counter() - start) * 1000, 2)
print({"surface": surface, "probabilities": probabilities, "model_version": bundle["version"], "elapsed_ms": elapsed_ms})

{'surface': 'dry', 'probabilities': {'dry': 0.9872, 'wet': 0.0017, 'dirt': 0.0112, 'snow': 0.0}, 'model_version': 'logreg-1.0', 'elapsed_ms': 3.75}


## 9. Журнал прогнозов: тип `jsonb` в PostgreSQL

Ответ ушел клиенту. Зачем его хранить?
- **разбор жалоб:** «машина затормозила на сухой дороге» – что пришло на вход и что ответила модель;
- **переобучение:** признаки и настоящее покрытие от водителей – новые размеченные данные;
- **наблюдение за качеством:** растет ли доля ошибок со временем, какая версия модели лучше;
- **поиск сбоев датчиков** по странным значениям признаков.

Хранить нужно вход, ответ, вероятности и версию модели. Признаков 14, и набор может поменяться вместе с моделью. Можно завести 14 столбцов, но каждая новая версия модели потребует менять схему. Вместо этого признаки хранятся **одним столбцом типа `jsonb`** – двоичного формата JSON в PostgreSQL:

```sql
CREATE TABLE IF NOT EXISTS predictions (
    id            integer GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    created_at    timestamptz NOT NULL DEFAULT now(),
    features      jsonb NOT NULL,        -- {"speed": 44.8, "tires": "летние", ...}
    surface       text NOT NULL,
    probabilities jsonb NOT NULL,        -- {"dry": 0.9872, "wet": 0.0017, ...}
    model_version text NOT NULL,
    elapsed_ms    double precision NOT NULL,
    actual        text,                  -- обратная связь (раздел 10)
    feedback_at   timestamptz
);
```

| Тип | Как хранит | Когда выбирать |
| --- | --- | --- |
| `json` | текст как есть: пробелы, порядок ключей, повторы | почти никогда |
| `jsonb` | разобранное двоичное значение: ключи без повторов, быстрый доступ по ключу, индексы | по умолчанию |

Операторы для `jsonb`:

| Запись | Результат |
| --- | --- |
| `features -> 'air_temp'` | значение по ключу как `jsonb` |
| `features ->> 'air_temp'` | значение по ключу как текст |
| `(features ->> 'air_temp')::numeric` | текст превращен в число: можно сравнивать и считать |
| `features @> '{"tires": "зимние"}'` | содержит ли объект такую пару ключ – значение |

Цена гибкости: база не проверяет типы и диапазоны внутри JSON (это делает Pydantic до записи), а запросы по признакам длиннее.

В Python словарь для столбца `jsonb` передается в обертке `Jsonb`: так psycopg знает, в каком виде отправить значение. Обратно psycopg сам превращает `jsonb` в словарь. Ячейки ниже работают во временной таблице `demo_predictions` тестовой базы (нужен сервер PostgreSQL и файл `.env`).

In [ ]:
import psycopg
from psycopg.rows import dict_row
from psycopg.types.json import Jsonb
from app.config import settings

conn = psycopg.connect(settings.test_database_url, row_factory=dict_row, autocommit=True)
conn.execute("DROP TABLE IF EXISTS demo_predictions")
conn.execute('''CREATE TABLE demo_predictions (
    id integer GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    features jsonb NOT NULL, surface text NOT NULL, probabilities jsonb NOT NULL,
    model_version text NOT NULL, actual text)''')

for item in reference["examples"]:
    surface, probabilities = predict_one(bundle, item["features"])
    row = conn.execute(
        '''INSERT INTO demo_predictions (features, surface, probabilities, model_version)
           VALUES (%(features)s, %(surface)s, %(probabilities)s, %(version)s) RETURNING id''',
        {"features": Jsonb(item["features"]), "surface": surface,
         "probabilities": Jsonb(probabilities), "version": bundle["version"]},
    ).fetchone()
print("записано прогнозов:", row["id"])

back = conn.execute("SELECT features, probabilities FROM demo_predictions WHERE id = 1").fetchone()
print(type(back["features"]), back["features"]["tires"], back["probabilities"])

In [ ]:
rows = conn.execute('''
    SELECT id, surface,
           features ->> 'air_temp' AS air_temp_text,
           (features ->> 'air_temp')::numeric AS air_temp,
           probabilities -> 'snow' AS p_snow
    FROM demo_predictions
    WHERE (features ->> 'air_temp')::numeric < 5
    ORDER BY id''').fetchall()
for row in rows:
    print(row)

print(conn.execute('''SELECT COUNT(*) AS n FROM demo_predictions
                      WHERE features @> '{"tires": "зимние"}' ''').fetchone())

## 10. Обратная связь и доля верных прогнозов

Водитель знает, по какой дороге ехал, и может отметить: «на самом деле был снег». Адрес `POST /api/predictions/{prediction_id}/feedback` записывает настоящее покрытие в столбец `actual` (задание З6):

```sql
UPDATE predictions SET actual = %(actual)s, feedback_at = now()
WHERE id = %(id)s
RETURNING id, surface, actual, surface = actual AS correct
```

`RETURNING` возвращает и вычисленный столбец: `surface = actual` – логическое значение «прогноз верен». Если строки с таким номером нет, `fetchone()` вернет `None`: ответ 404.

Доля верных прогнозов считается **одним запросом SQL**, без цикла по строкам в Python:
- `COUNT(actual)` считает только непустые значения – это число прогнозов с обратной связью;
- `COUNT(*) FILTER (WHERE actual = surface)` – число верных (как в З2 занятия Т4.Б1.4);
- `NULLIF(x, 0)` возвращает `NULL`, если `x` равен нулю: деления на ноль не будет, результат будет `NULL`;
- `::numeric` у делимого: деление целых чисел в SQL отбрасывает дробную часть (`3 / 4` дает 0).

In [ ]:
ACCURACY = '''
    SELECT COUNT(actual) AS with_feedback,
           COUNT(*) FILTER (WHERE actual = surface) AS correct,
           ROUND(COUNT(*) FILTER (WHERE actual = surface)::numeric / NULLIF(COUNT(actual), 0), 3) AS accuracy
    FROM demo_predictions'''
print("до обратной связи:", conn.execute(ACCURACY).fetchone())

# водители отметили первые восемь прогнозов: настоящее покрытие из эталона
for number, item in enumerate(reference["examples"][:8], start=1):
    conn.execute("UPDATE demo_predictions SET actual = %s WHERE id = %s", [CODES[item["label"]], number])
print("после:            ", conn.execute(ACCURACY).fetchone())
print("целочисленное деление в SQL: 3 / 4 =", conn.execute("SELECT 3 / 4 AS r").fetchone()["r"])

In [ ]:
conn.execute("DROP TABLE demo_predictions")
conn.close()

Доля верных по обратной связи и качество на тестовой выборке – разные числа. Тестовая выборка взята из тех же данных, что и обучающая. Обратная связь приходит с реальных дорог: другие сезоны, другие автомобили, данные меняются со временем. Если доля верных по обратной связи падает, модель пора переобучить на новых данных из журнала. И еще одно смещение: водители охотнее сообщают об ошибках, чем о верных прогнозах, поэтому доля по обратной связи бывает заниженной.

## 11. Сервис целиком: запросы клиента

Запустите решение или свою работу (`fastapi dev app/main.py` в папке `server`) и выполните ячейку: так с сервисом работает любая программа на Python. Страница на React и мобильное приложение делают то же самое на своих языках.

In [ ]:
import requests

BASE = "http://127.0.0.1:8000"
try:
    print(requests.get(f"{BASE}/api/model", timeout=3).json()["version"])
    ok = requests.post(f"{BASE}/api/predict", json=example["features"], timeout=3)
    print(ok.status_code, ok.json())
    bad = requests.post(f"{BASE}/api/predict", json={**example["features"], "road_temp": -999, "rec_speed": 90}, timeout=3)
    print(bad.status_code, bad.json())
    number = ok.json().get("id")
    if number:
        print(requests.post(f"{BASE}/api/predictions/{number}/feedback", json={"actual": "dry"}, timeout=3).json())
        print(requests.get(f"{BASE}/api/predictions/accuracy", timeout=3).json())
except requests.ConnectionError:
    print("сервер не запущен: fastapi dev app/main.py в папке ПР_ML_сервис/server")

## 12. Итоги

1. Модель из ноутбука становится сервисом в три шага: сохранить конвейер целиком в файл, загрузить его один раз при старте сервера, обернуть программным интерфейсом с проверкой входных данных.
2. Файл `joblib` (как и `pickle`) может выполнить код при загрузке: загружайте только свои модели.
3. Версии scikit-learn, numpy, pandas, joblib в сервисе должны совпадать с версиями при обучении. Сервис сверяет их при старте и останавливается с понятной ошибкой.
4. Модель ответит на любые числа, поэтому проверка входных данных обязательна: нет признака, неверный тип, значение вне диапазона – ответ 422 с именем признака.
5. Модель Pydantic заменяет ручную проверку объявлением полей и бесплатно дает описание в документации.
6. Ответ сервиса: класс, вероятности всех классов, версия модели, время. Порядок вероятностей задает `pipeline.classes_`.
7. Каждый прогноз записывается в журнал: признаки и вероятности – в столбцах `jsonb`. Обратная связь водителей дает долю верных прогнозов на реальных дорогах и данные для переобучения.

**Словарь**

| Термин | Значение |
| --- | --- |
| сериализация | запись объекта в байты или файл, чтобы потом восстановить его |
| pickle, joblib | модули Python для сериализации объектов; joblib быстрее для массивов numpy |
| паспорт модели | сведения о модели рядом с ней: версия, признаки, классы, версии библиотек, метрики |
| функция жизненного цикла | функция `lifespan`: код до `yield` – при старте сервера, после – при остановке |
| `app.state` | хранилище общих объектов приложения FastAPI |
| строгий режим | режим проверки Pydantic, в котором значения не превращаются из одного типа в другой |
| `jsonb` | тип PostgreSQL для JSON в разобранном двоичном виде |
| `->`, `->>` | операторы `jsonb`: значение по ключу как `jsonb` и как текст |
| обратная связь | сведения о настоящем ответе, которые присылает пользователь после прогноза |
| сдвиг данных | изменение данных со временем, из-за которого качество модели падает |

**Вопросы для самопроверки**

1. Почему сохраняют весь конвейер, а не только логистическую регрессию?
2. Что может случиться при загрузке файла модели, присланного незнакомцем?
3. Какие три исхода возможны при загрузке модели в другой версии scikit-learn? Какой из них опаснее всего?
4. Где и когда загружается модель в сервисе? Почему не в обработчике?
5. Назовите три вида ошибок входных данных и код ответа для каждого.
6. Почему `True` проходит проверку `isinstance(value, int)`?
7. Что дает строгий режим Pydantic и почему он включен только у числовых полей?
8. В каком порядке `predict_proba` возвращает вероятности?
9. Зачем в ответе и в журнале версия модели?
10. Чем `jsonb` удобнее 14 отдельных столбцов? Чем приходится платить?

**Задачи**

1. Подставьте в пример из раздела 6 значения `air_temp = 150` и `slip = 80` (тоже коды ошибок из итоговой работы). Как меняются вероятности? Какой признак опаснее для беспилотного автомобиля?
2. Измерьте время загрузки модели в свежем процессе Python: команда `python -c "import time; t = time.perf_counter(); import joblib; joblib.load('model/model.joblib'); print(time.perf_counter() - t)"` в папке `server`. Сколько прогнозов можно сделать за это время?
3. Допишите модель `Demo` из раздела 7 до 5 признаков на выбор и посмотрите ее описание `model_json_schema()`.
4. Напишите запрос SQL: средняя вероятность ответа модели (`(probabilities ->> surface)::numeric`) по каждому покрытию. На каких покрытиях модель увереннее?
5. Напишите запрос SQL: доля верных прогнозов отдельно для каждой версии модели (`GROUP BY model_version`).